# Pytorch Basics

In [ ]:
!nvidia-smi

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
import numpy as np
from matplotlib import pyplot as plt

## Basic Operations

In [ ]:
# reminder this is a numpy array, of integers
arr = np.array([[1, 2, 3], [4, 5, 6]])
print(arr)
print(arr.dtype)

In [ ]:
# tensors can be created from lists, arrays etc.
# they look a lot like numpy arrays
tensor1 = torch.tensor([[0., -2., 3.], [12., 5., 1.]])
print(tensor1)
print(tensor1.dtype)

tensor2 = torch.tensor(arr)
print(tensor2)
print(tensor2.dtype)

In [ ]:
# tensors are mutable, i.e. we can modify them
tensor2[0,0] = 1000
print(tensor2)

In [ ]:
# we can do stuff like add tensors
print(tensor1)
print("+")
print(tensor2)
print("=")
print(tensor1 + tensor2)

In [ ]:
# we can even add tensors and numpy arrays.
# this results in a tensor, but may lead to some unexpected behavior.
tensor1 + arr

In [ ]:
# multiply scalars and tensors...
5*tensor1

In [ ]:
# multiply tensors (elementwise/multiplication)
print(tensor1)
print("*")
print(tensor2)
print("=")
print(tensor1 * tensor2)

In [ ]:
# ...matrix multiplication...
# whoops! the shapes don't match :)
torch.matmul(tensor1,  tensor1)

In [ ]:
# we can transpose tensors... switch axis 0 and 1.
tensor1.transpose(0, 1)

In [ ]:
# this works
torch.matmul(tensor1, tensor1.transpose(0, 1))

In [ ]:
# @ symbol gives the same result
tensor1 @ tensor1.transpose(0, 1)

In [ ]:
tensor1.transpose(0, 1) @ tensor1

In [ ]:
# we can also reshape tensors. this function is called "view" in pytorch
print("shape (2, 3)")
print(tensor1)
print()
print("shape (6,)")
print(tensor1.view(6))
print()
print("shape (6, 1)")
print(tensor1.view(6, 1))  # not the same!!
print()
print("shape (1, 6)")
print(tensor1.view(1, 6))  # not the same!!
print()
print("shape (3, 2)")
print(tensor1.view(3, 2))

In [ ]:
# the shape has to fit the number of elements
tensor1.view(5, 2)

In [ ]:
# many simple mathematical functions can be applied elementwise.
print(tensor2)
# ...exp(1000) is not a good idea
print(torch.exp(tensor2))

In [ ]:
print(tensor1)
print(torch.sin(tensor1))

In [ ]:
# summary functions like sum, mean, max...
print("original tensor")
print(tensor1)
print()
print("mean")
print(tensor1.mean())
print()
print("sum")
print(tensor1.sum())
print()
print("max")
print(tensor1.max())
print()

print("mean over axis 1")
print(tensor1.mean(axis=1))
print()
print("max over axis 0")
print(tensor1.max(axis=0))

In [ ]:
# tensors can be converted to numpy!
tensor2.numpy()

## Creating Tensors

In [ ]:
# besides converting lists or arrays, we can...

In [ ]:
# ...use linspace
x = torch.linspace(0, 2*np.pi, 1000)
y = torch.sin(x)
plt.plot(x, y)
plt.show()

In [ ]:
# create specific values
zeros = torch.zeros(2, 5, 3)  # tensors can have more than 2 dimensions
print(zeros)
print()
ones = torch.ones(5)
print(ones)

In [ ]:
# create random values
print(torch.rand(3, 1))  # uniform in range [0, 1)
print()
print(torch.randn(4, 2))  # standard normal distribution

In [ ]:
# more sophisticated
print(torch.distributions.Uniform(-1, 1).sample())
print()
print(torch.distributions.Uniform(-1, 1).sample((1,)))  # not the same!
print()
print(torch.distributions.Uniform(-1, 1).sample((2,4)))
print()
print(torch.distributions.Uniform(-10, 10).sample((2,4)))

## GPU Support

This whole section doesn't really work if you don't have a CUDA GPU available... If running this on Google Colab, make sure you have selected a GPU runtime.

In [ ]:
# tensors are by default on the CPU
print(tensor1)
print(tensor1.device)

In [ ]:
# IF you have a GPU available -- you can use it!
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device is", device)

In [ ]:
# tensors needs to be moved to the device explicitly
tensor = torch.randn(2, 5)
tensor_gpu = tensor.to(device)
print(tensor.device, tensor_gpu.device)

In [ ]:
print(tensor)
print(tensor_gpu)

In [ ]:
# any operations on GPU tensors will be on GPU as well
exp = torch.exp(tensor_gpu)
print(exp)

In [ ]:
# oh no! this doesn't work anymore with GPU tensors
exp.numpy()

In [ ]:
# we need to move back to cpu explicitly
exp.cpu().numpy()

In [ ]:
# or if you know what you are doing:
exp.numpy(force=True)

In [ ]:
# tensors we want to operate on have to be on the same device
tensor + tensor_gpu

In [ ]:
tensor.to(device) + tensor_gpu

In [ ]:
# we can also create tensors on a device directly
torch.ones(5, device=device)

## Automatic Differentiation

In [ ]:
# first tell torch that we're gonna need gradients for this thing
x = torch.tensor(3., requires_grad=True)

In [ ]:
# do all the computations we want
y = x**2

In [ ]:
# and backpropagate!
y.backward()

In [ ]:
# voila!
print(x.grad)

In [ ]:
# let's do it again!
# but this seems wrong...
y = x**2
y.backward()
print(x.grad)

In [ ]:
# we have to explicitly delete the gradient
x.grad.zero_()

In [ ]:
# now it works again
y = x**2
y.backward()
print(x.grad)

In [ ]:
# be careful...
x.grad.zero_()
y = x**2
y.backward()
gradient = x.grad
x.grad.zero_()
print(gradient)

# we lost our gradient! oh no

In [ ]:
# we gotta do stuff with it before zeroing out
tensor = torch.tensor(2.)

print("Before", tensor)
y = x**2
y.backward()
gradient = x.grad
tensor += gradient

x.grad.zero_()
print("After", tensor)
# we succesfully added the gradient! 
# zeroing it out no longer matters after that

In [ ]:
# or copy the data first
x.grad.zero_()
y = x**2
y.backward()
gradient = x.grad.clone()
x.grad.zero_()
print(gradient)

In [ ]:
# we can go more complex...
x = torch.tensor(3., requires_grad=True)
y = x**2
z = y**3
z.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# z = x**6, so the derivative should be 6 * x**5
6 * 3**5

In [ ]:
# we can go as complex as we like!
x = torch.tensor(2., requires_grad=True)
out = torch.sqrt(torch.exp(torch.cos(x**2) + 1)*5)
out.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# for ML use cases, we usually have vector-valued variables
w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)

dotproduct = (w*x).sum()
dotproduct.backward()
print("x", x)
print("w", w)
print()
print("dotproduct", dotproduct)
print()
print("dotproduct gradient", w.grad)
w.grad.zero_()

In [ ]:
# we can also compute multiple gradients in parallel!

w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)
w2 = torch.tensor(0.5, requires_grad=True)

dotproduct = (w*x).sum()
scaled = w2*dotproduct
scaled.backward()
print("x", x)
print("w", w)
print("w2", w2)
print()
print("dotproduct", dotproduct)
print("scaled", scaled)
print()
print("dotproduct gradient for w", w.grad)
print("dotproduct gradient for w2", w2.grad)
w.grad.zero_()
w2.grad.zero_()

In [ ]:
# all operations have to be differentiable!
# things like equality aren't
w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)

equality = (w == x).sum()
equality.backward()
print("x", x)
print("w", w)
print()
print("dotproduct", dotproduct)
print()
print("dotproduct gradient", w.grad)
w.grad.zero_()

In [ ]:
# everything has to happen with torch operations.
# you cannot convert to numpy or something like that!
# actually the numpy() call will fail, good thing!
w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)

dotproduct = (w.numpy() * x).sum()
dotproduct.backward()
print("x", x)
print("w", w)
print()
print("dotproduct", dotproduct)
print()
print("dotproduct gradient", w.grad)
w.grad.zero_()

In [ ]:
# we could try forcing it...
w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)

dotproduct = (torch.tensor(w.numpy(force=True)) * x).sum()
dotproduct.backward()
print("x", x)
print("w", w)
print()
print("dotproduct", dotproduct)
print()
print("dotproduct gradient", w.grad)
w.grad.zero_()

In [ ]:
# an aside -- for tensors with a gradient, you have to do this to convert to numpy (or use force=True)
w.detach().numpy()

In [ ]:
# it also doesn't work on x.
# although the error message is somewhat misleading, as x doesn't require a gradient...
w = torch.randn(5, requires_grad=True)
x = torch.arange(1, 6, dtype=torch.float32)

dotproduct = (w * x.numpy()).sum()
dotproduct.backward()
print("x", x)
print("w", w)
print()
print("dotproduct", dotproduct)
print()
print("dotproduct gradient", w.grad)
w.grad.zero_()

In [ ]:
# note, during our computation, we can overwrite the Python variables.
# here we use y multiple times.
x = torch.tensor(3., requires_grad=True)
y = x**2
y = y**3
y.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# but do NOT overwrite the variables you want gradients for!
x = torch.tensor(3., requires_grad=True)
x = x**2
y = x**3
y.backward()
print(x.grad)
#x.grad.zero_()

In [ ]:
# the same error above also happens if we try to compute gradients for intermediate steps
x = torch.tensor(3., requires_grad=True)
y = x**2
z = y**3
z.backward()
print("y requires grad?", y.requires_grad)
print(y.grad)
#y.grad.zero_()

In [ ]:
# now it works :)
# use retain_grad()
x = torch.tensor(3., requires_grad=True)
y = x**2
y.retain_grad()
z = y**3
z.backward()
print("y requires grad?", y.requires_grad)
print(y.grad)
y.grad.zero_()

In [ ]:
# if you ever want to do some stuff "on the side" without tracking gradients, that's possible too!
# we can go more complex...
x = torch.tensor(3., requires_grad=True)
y = x**2

with torch.no_grad():
    random_side_stuff = torch.exp(y)**2

z = y**3
z.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# the thing is, since that stuff is usually "on the side" anyway, it won't be part of our computation graph.
# so even if we don't do this with no_grad(), it doesn't make our program incorrect.
# however, it's good practice to include no_grad() to prevent unnecessary tracking of operations by Pytorch.
# we will also see some situations later where you actually *need* to do no_grad()!
x = torch.tensor(3., requires_grad=True)
y = x**2

random_side_stuff = torch.exp(y)**2

z = y**3
z.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# here's one example with the GPU. this will crash if you have gpu support i.e. device==cuda!
x = torch.tensor(3., requires_grad=True).to(device)
y = x**2
y.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# the easiest way would be to just specify the device when creating the tensor
x = torch.tensor(3., requires_grad=True, device=device)
y = x**2
y.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# but maybe the device conversion only comes later/is optional
x = torch.tensor(3., requires_grad=True)
# ...
x = x.to(device)
y = x**2
y.backward()
print(x.grad)
x.grad.zero_()

In [ ]:
# in that case, this is ok, although a bit awkward
x = torch.tensor(3., requires_grad=True)
with torch.no_grad():
    x = x.to(device)
x.requires_grad = True

y = x**2
y.backward()
print(x.grad)
x.grad.zero_()